# Visualize Signifiance Before and After Covariate Adjustments

In [ ]:
import re

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from hk1_r12ter_analysis.config import (
    GENE_ID,
    PROCESSED_DATA_DIR,
    PROJ_ROOT,
    REPORTS_DIR,
    logger,
)
from hk1_r12ter_analysis.data.transform import transform_log10
from hk1_r12ter_analysis.io import (
    load_dataframe_from_file,
    make_filename,
    save_dataframe_to_file,
)
from hk1_r12ter_analysis.modeling.linear_model import (
    significance_for_covariate_adjustments,
)
from hk1_r12ter_analysis.util import ceil_to_decimal, multiple_str_replace

### Load annotations for mapping SNP IDs to shorthands for file/directory names

In [ ]:
input_dirpath = PROCESSED_DATA_DIR / "REDS"
input_filename = make_filename("REDS", "Genomics", GENE_ID, "Metadata")
df_genomics_ann = load_dataframe_from_file(input_dirpath / input_filename, index_col="ID")
# Change periods to semicolons when using RSID in file/directory name
df_rsid_map = df_genomics_ann["RSID"].replace(":", ".")
df_rsid_map

## REDS Recall Data

In [ ]:
sample_key = "RECALL DONOR ID"
group_key = "Day"
source_type = "REDS-Recall"
data_types = ["Metabolomics"]

alpha = 0.05  # Value to determine of significance
genotype = "i_rs10762276:71052469:C:T"
control_group = 0  # Homozygous reference
# REDS covariates: Additive, Sex, Age, and BMI
covariates = {"AS", "Sex", "Age", "BMI"}
contrast = (0, 2)  # (0, 1), (0, 2), (1, 2)
use_fdr = True
filetype = "pdf"

if group_key is not None:
    covariates.add(group_key)


data_dirpath = (
    PROCESSED_DATA_DIR
    / "REDS"
    / "LinearModel"
    / "-".join([source_type] + data_types)
    / "-".join(["Covariates"] + sorted(covariates))
)

figure_dirpath = (
    REPORTS_DIR
    / "REDS"
    / "LinearModel"
    / "-".join([source_type] + data_types)
    / "-".join(["Covariates"] + sorted(covariates))
)
figure_dirpath.mkdir(parents=True, exist_ok=True)

### Determine significance at given level

In [ ]:
filename_args = [GENE_ID, df_rsid_map.get(genotype, genotype)]
if contrast is not None:
    filename_args += ["{0}_vs_{1}".format(*contrast)]
input_filename = make_filename(*filename_args, "pvalues")
df_pvalues = load_dataframe_from_file(data_dirpath / input_filename, index_col=0)
pvalue_str = "FDR" if use_fdr else "pvalue"
columns = df_pvalues.columns[df_pvalues.columns.str.startswith(pvalue_str)]
df_significance = significance_for_covariate_adjustments(df_pvalues.loc[:, columns], alpha)
df_significance.loc[:, columns] = -transform_log10(df_significance.loc[:, columns])
df_significance = df_significance.rename(
    {col: col.replace(pvalue_str, f"-log10({pvalue_str})") for col in columns}, axis=1
)

# Save file
filename_args = [GENE_ID, df_rsid_map.get(genotype, genotype)]
if contrast is not None:
    filename_args += ["{0}_vs_{1}".format(*contrast)]
output_filename = make_filename(*filename_args, pvalue_str, f"Sig_{alpha}")
save_dataframe_to_file(df_significance, data_dirpath / output_filename, index=True)
df_significance

### Visualization options

In [ ]:
palette = {
    "Significant": "xkcd:blue",
    "Sig. when adjusted": "xkcd:green",
    "Non-sig. when adjusted": "xkcd:red",
    "Non-significant": "xkcd:light grey",
}
fancy_str_replacements = {"pvalue": "$p$-value", "log10": "-log$_{10}$"}
scatter_kwargs = dict(edgecolor="black", zorder=4, s=50, alpha=0.75)
pvalue_lines_kwargs = dict(
    color="xkcd:black",
    linestyle="--",
    linewidth=2,
)
diag_line_kwargs = dict(
    color="xkcd:grey",
    linestyle=":",
    linewidth=2,
)
title_fontdict = {"size": "large", "weight": "bold"}
label_fontdict = {"size": "large"}
tick_fontdict = {"size": "medium"}

lim_factor = 3
dec = 0
equal_xylim = True

### Load data and visualize

In [ ]:
title_str = (
    f"{GENE_ID} {genotype}\n"
    f" Correlates to {', '.join(data_types) if len(data_types) > 2 else ' and '.join(data_types)}\n"
    f"in {source_type.replace('-', ' ')} Donors"
)
x, y, hue = tuple(df_significance.columns)

absmin = -0.1
xmax = ceil_to_decimal(df_significance.max()[x] / lim_factor, dec) * lim_factor
ymax = ceil_to_decimal(df_significance.max()[y] / lim_factor, dec) * lim_factor
absmax = max(xmax, ymax)

fig, ax = plt.subplots(
    nrows=1,
    ncols=1,
    figsize=(5, 5),
)

if bool(pvalue_lines_kwargs):
    ax.hlines(
        -np.log10(alpha),
        xmin=absmin,
        xmax=absmax if equal_xylim else xmax,
        **pvalue_lines_kwargs,
    )
    ax.vlines(
        -np.log10(alpha),
        ymin=absmin,
        ymax=absmax if equal_xylim else ymax,
        **pvalue_lines_kwargs,
    )

if bool(diag_line_kwargs):
    ax.plot(
        [absmin, absmax],
        [absmin, absmax],
        label="Unadjusted = Covariate Adjusted",
        **diag_line_kwargs,
    )

ax = sns.scatterplot(
    data=df_significance,
    x=x,
    y=y,
    hue=hue,
    # size=None,
    # style=None,
    palette=palette,
    # hue_order=None,
    # hue_norm=None,
    # sizes=None,
    # size_order=None,
    # size_norm=None,
    # markers=True,
    # style_order=None,
    legend="auto",
    ax=ax,
    **scatter_kwargs,
)

ax.set_title(title_str, fontdict=title_fontdict)
ax.set_xlabel(multiple_str_replace(x, fancy_str_replacements), fontdict=label_fontdict)
ax.set_ylabel(multiple_str_replace(y, fancy_str_replacements), fontdict=label_fontdict)
ax.set_xticks(ax.get_xticks(), labels=ax.get_xticklabels(), fontdict=tick_fontdict)
ax.set_yticks(ax.get_yticks(), labels=ax.get_yticklabels(), fontdict=tick_fontdict)
ax.set_xlim(absmin, absmax if equal_xylim else xmax)
ax.set_ylim(absmin, absmax if equal_xylim else ymax)
handles, labels = ax.get_legend_handles_labels()
value_counts = df_significance["Significance"].value_counts()
labels = [
    f"{label} (n={value_counts.loc[label]})" if label in value_counts else label
    for label in labels
]
ax.legend(
    handles=handles,
    labels=labels,
    loc="center",
    bbox_to_anchor=(0.5, -0.25),
    ncols=ceil_to_decimal(len(palette) / 4),
    frameon=True,
    fancybox=True,
    edgecolor="xkcd:black",
    labelspacing=0.1,
    handletextpad=0.1,
    fontsize="medium",
)

sns.despine(ax=ax)
output_filename = make_filename(*filename_args, pvalue_str, f"Sig_{alpha}", filetype=filetype)
fig.savefig(figure_dirpath / output_filename)
plt.show()
plt.close()
df_significance.sort_values(by=f"-log10({pvalue_str}) covariate adjustment", ascending=False).head(
    20
)